# KM-18: PyTorch vs TensorRT FP32 (full pipeline)
GPU Colab, lima model FP32, satu video dan config. Urutan: siapkan input lokal → ekspor/reuse engine → jalankan `.pt` dan `.engine` bergantian → bandingkan event. Ekspor dan penyalinan ke Drive tidak masuk durasi pipeline. Upload notebook ini dan `hauling/` terbaru ke `MyDrive/hauling_colab/`.


In [ ]:
import os, subprocess, sys
assert sys.version_info >= (3, 11), 'Python >=3.11 diperlukan'
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', 'torch==2.12.1', 'torchvision==0.27.1',
    '--index-url', 'https://download.pytorch.org/whl/cu126',
], check=True)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics==8.4.145',
    'opencv-python==5.0.0.93', 'numpy==2.4.6', 'lap==0.5.13', 'PyYAML==6.0.3',
    'onnx>=1.12,<2', 'tensorrt-cu12',
], check=True)
os.environ.update(YOLO_OFFLINE='true', YOLO_AUTOINSTALL='false', YOLO_CONFIG_DIR='/content/hauling-tensorrt-runtime', YOLO_VERBOSE='false')
import torch, tensorrt, ultralytics
assert torch.cuda.is_available(), 'Ganti runtime Colab ke GPU'
print('GPU:', torch.cuda.get_device_name(0), '| CUDA:', torch.version.cuda, '| PyTorch:', torch.__version__, '| TensorRT:', tensorrt.__version__, '| Ultralytics:', ultralytics.__version__)


## Salin input ke disk lokal Colab
Proses video dari `/content`, bukan langsung dari Drive. Jangan jalankan ulang sel ini jika folder kerja sudah ada.


In [ ]:
from google.colab import drive
from pathlib import Path
import shutil
drive.mount('/content/drive')
BASE = Path('/content/drive/MyDrive/hauling_colab')
WORK = Path('/content/hauling-tensorrt-km18')
required = ('hauling/detector.py', 'hauling/pipeline.py', 'hauling/summary.py',
            'hauling/comparison.py', 'model/Hauling_Vehicle_yolov8m_SingleCls_V11.pt')
assert all((BASE / name).is_file() for name in required), 'Upload hauling/ terbaru dan model dahulu'
assert ('def validate_model_path' in (BASE / 'hauling/detector.py').read_text()
        and 'model_runtime' in (BASE / 'hauling/pipeline.py').read_text()), 'hauling/ di Drive masih versi lama; replace seluruh folder hauling/ dahulu'
assert not WORK.exists(), f'{WORK} sudah ada; jangan timpa data run lama'
shutil.copytree(BASE, WORK, ignore=shutil.ignore_patterns('outputs', 'results_trt', 'engines', '*.ipynb'))
sys.path.insert(0, str(WORK))
print('Input lokal:', WORK, '| ruang tersedia:', round(shutil.disk_usage(WORK).free / 1024**3, 1), 'GiB')


## Ekspor atau gunakan ulang lima engine FP32
`ENGINE_SOURCE = None` mengekspor lima engine baru ke folder Drive baru; isi nama folder di `engines/` untuk memakai ulang engine setelah GPU, versi, bobot, parameter, dan checksum cocok. Tidak ada overwrite. Dua classifier memakai dynamic batch hingga 32 karakter; `half=False` berarti FP32.


In [ ]:
import hashlib, json, time
from datetime import datetime, timezone
from ultralytics import YOLO
ENGINE_SOURCE = None  # nama folder fp32_* di engines/ untuk reuse; None untuk ekspor baru
MODELS = {
    'vehicle': ('Hauling_Vehicle_yolov8m_SingleCls_V11', 640, False),
    'digit_detector': ('character_angka_detection_hauling_yolov8m_V2', 224, False),
    'letter_detector': ('character_huruf_detection_hauling_yolov8m_V2', 224, False),
    'digit_classifier': ('classification_hauling_angka_yolo8m_imgz96_v9', 96, True),
    'letter_classifier': ('classification_hauling_huruf_yolo8m_imgz96_v9', 96, True),
}
def sha256(path):
    with path.open('rb') as stream:
        return hashlib.file_digest(stream, 'sha256').hexdigest()

environment = {
    'gpu': torch.cuda.get_device_name(0),
    'capability': list(torch.cuda.get_device_capability(0)),
    'driver': subprocess.check_output(
        ['nvidia-smi', '--query-gpu=driver_version', '--format=csv,noheader'], text=True
    ).strip(),
    'cuda': torch.version.cuda, 'torch': torch.__version__,
    'tensorrt': tensorrt.__version__, 'ultralytics': ultralytics.__version__,
    'precision': 'fp32',
}
model_info = {}
for key, (stem, imgsz, dynamic) in MODELS.items():
    weights = WORK / 'model' / f'{stem}.pt'
    assert weights.is_file(), f'Model hilang: {weights}'
    model_info[key] = {
        'weights': weights.name, 'weights_sha256': sha256(weights),
        'imgsz': imgsz, 'dynamic': dynamic, 'batch': 32 if dynamic else 1,
    }
if ENGINE_SOURCE is not None:
    stored = BASE / 'engines' / ENGINE_SOURCE
    assert stored.parent == BASE / 'engines' and stored.is_dir(), 'ENGINE_SOURCE harus nama folder di engines/'
    saved = json.loads((stored / 'manifest.json').read_text())
    assert saved['environment'] == environment, 'GPU/driver/versi/presisi berbeda: ekspor ulang'
    for key, info in model_info.items():
        entry = saved['models'][key]
        assert all(entry[field] == value for field, value in info.items()), f'Bobot/parameter berubah: {key}'
        engine = stored / f"{MODELS[key][0]}.engine"
        assert engine.is_file() and sha256(engine) == entry['engine_sha256'], f'Engine rusak: {engine}'
        target = WORK / 'model' / engine.name
        assert not target.exists(), f'Engine lokal sudah ada: {target}'
    for stem, _, _ in MODELS.values():
        engine = stored / f'{stem}.engine'
        target = WORK / 'model' / engine.name
        shutil.copy2(engine, target)
    print('Menggunakan engine tervalidasi:', stored)
else:
    for key, (stem, imgsz, dynamic) in MODELS.items():
        weights = WORK / 'model' / f'{stem}.pt'
        engine = weights.with_suffix('.engine')
        assert not engine.exists(), f'Engine sudah ada; jangan timpa: {engine}'
        started = time.monotonic()
        exported = Path(YOLO(str(weights)).export(
            format='engine', device=0, imgsz=imgsz, half=False,
            simplify=False, dynamic=dynamic, batch=32 if dynamic else 1,
        ))
        assert exported == engine and engine.is_file(), f'Ekspor gagal: {key}'
        model_info[key].update(
            engine_sha256=sha256(engine), engine_bytes=engine.stat().st_size,
            export_seconds=round(time.monotonic() - started, 3),
        )
        print(key, '->', engine.name, model_info[key]['export_seconds'], 's')
    stored = BASE / 'engines' / f"fp32_{datetime.now(timezone.utc):%Y%m%dT%H%M%S%fZ}"
    stored.mkdir(parents=True, exist_ok=False)
    for stem, _, _ in MODELS.values():
        shutil.copy2(WORK / 'model' / f'{stem}.engine', stored)
    (stored / 'manifest.json').write_text(json.dumps({'environment': environment, 'models': model_info}, indent=2) + '\n')
    print('Engine dan metadata tersimpan:', stored)


`.pt` → ONNX (Ultralytics) → TensorRT Builder → `.engine`. Optimasi/kernel dipilih oleh TensorRT untuk GPU saat build, bukan oleh `hauling/`. Tidak ada kalibrasi INT8 atau fallback otomatis ke `.pt`. Engine dari GPU/runtime lain harus diperiksa sebelum digunakan.


## Jalankan dua runtime lalu bandingkan event
`MAX_FRAMES=None` memproses sampai EOF; 300 hanya untuk smoke test. Keduanya memakai video, config, GPU, dan opsi sama. Run membuat video anotasi dan dump OCR; setelah run selesai, notebook membuat crop dan perbedaan event. Output disimpan dengan nama unik; FPS di video adalah nilai berjalan, bukan rata-rata total.


In [ ]:
import json, shutil, subprocess, sys
from datetime import datetime, timezone
from hauling.comparison import compare_runs, save_plate_evidence
VIDEO = '20261007133257-3201.mp4'
MAX_FRAMES = None  # ubah ke 300 hanya untuk smoke test, bukan laporan performa
assert VIDEO in {'20261007133257-3201.mp4', '00000018001003000.mp4', '00000018001003100.mp4'}
run_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
runs, run_dirs = {}, {}
for runtime, suffix in (('pytorch', '.pt'), ('tensorrt', '.engine')):
    output = WORK / 'outputs' / f'{VIDEO.removesuffix(".mp4")}_{runtime}_{run_id}'
    assert not output.exists(), f'Output sudah ada: {output}'
    destination = BASE / 'results_trt' / output.name
    assert not destination.exists(), f'Arsip sudah ada: {destination}'
    def model(key):
        return str(WORK / 'model' / f'{MODELS[key][0]}{suffix}')
    cmd = [
        sys.executable, '-m', 'hauling', '--video', str(WORK / 'videos' / VIDEO),
        '--vehicle-model', model('vehicle'), '--config', str(WORK / '_track_km18_01_V2.conf'),
        '--output-dir', str(output), '--device', '0', '--threads', '2', '--dump-plate-crops',
        '--plate-character-model', model('digit_detector'), '--plate-letter-model', model('letter_detector'),
        '--plate-digit-classifier', model('digit_classifier'), '--plate-letter-classifier', model('letter_classifier'),
    ]
    if MAX_FRAMES is not None:
        cmd.extend(['--max-frames', str(MAX_FRAMES)])
    completed = subprocess.run(cmd, cwd=WORK)
    if output.is_dir():
        shutil.copy2(stored / 'manifest.json', output / 'engine_manifest.json')
        shutil.copytree(output, destination)  # simpan juga output parsial bila run gagal
    completed.check_returncode()
    summary = json.loads((output / 'run_summary.json').read_text())
    assert (summary['status'] == 'completed' and summary['model_runtime'] == runtime
            and summary['video_written']), 'Run/video gagal; cek run_summary.json'
    if MAX_FRAMES is None:
        assert summary['stop_reason'] == 'eof', 'Video belum diproses sampai EOF'
    assert summary['ocr_attempts'] > 0, 'OCR belum teruji; gunakan cuplikan lebih panjang'
    assert (output / 'annotated.mp4').is_file() and (output / 'plate_debug').is_dir(), 'Bukti video/OCR hilang'
    save_plate_evidence(WORK / 'videos' / VIDEO, output)
    shutil.copytree(output / 'plates', destination / 'plates')
    runs[runtime] = summary
    run_dirs[runtime] = output
    print(runtime, summary['total_frames'], 'frames,', summary['duration_seconds'], 's,', summary['events'], 'events; saved:', destination)
assert runs['pytorch']['total_frames'] == runs['tensorrt']['total_frames'], 'Jumlah frame berbeda'
comparison = WORK / 'outputs' / f'{VIDEO.removesuffix(".mp4")}_comparison_{run_id}'
report = compare_runs(WORK / 'videos' / VIDEO, run_dirs['pytorch'], run_dirs['tensorrt'], comparison)
shutil.copytree(comparison, BASE / 'results_trt' / comparison.name)
print('Event berbeda:', report['different_event_frames'], '| bukti:', BASE / 'results_trt' / comparison.name)
print('FPS proses:', {key: round(value['total_frames'] / value['duration_seconds'], 2) for key, value in runs.items()})
print('Speedup (total pipeline):', round(runs['pytorch']['duration_seconds'] / runs['tensorrt']['duration_seconds'], 2), 'x')
for key in ('total_detections', 'unique_tracks', 'events', 'speed_valid', 'overspeed', 'unsafe_distance', 'plate_detected', 'ocr_decode_ok'):
    print(key, runs['pytorch'][key], runs['tensorrt'][key])
print('Timing tahap (detik):', {key: value['timing_seconds'] for key, value in runs.items()})
